In [1]:
import fitz
import re
from pathlib import Path

In [3]:
def normalize_pdf_text(text: str) -> str: #Xử lý văn bản trích xuất từ PDF để loại bỏ ký tự không mong muốn và chuẩn hóa định dạng
    if not text:
        return ""
    
    text = text.replace("\x00", " ")
    text = text.replace("\u00a0", " ")

     # Chuẩn hóa xuống dòng
    text = re.sub(r"\r\n|\r", "\n", text)
     # Xóa khoảng trắng thừa trong từng dòng
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]
    # Loại bỏ các dòng trống
    cleaned_lines = [] #danh sách để lưu các dòng đã được làm sạch
    prev_empty = False #dùng để nhớ dòng trước có rỗng không
    for line in lines:#
        if not line:
            if not prev_empty:
                cleaned_lines.append("")
            prev_empty = True
        else:
            cleaned_lines.append(line)
            prev_empty = False
    
    return "\n".join(cleaned_lines).strip()

def is_probably_scanned_pdf(pdf_path: str, min_chars_per_page: int = 80) -> bool:
    doc = fitz.open(pdf_path)

    # Nếu file không có trang nào thì coi như không trích xuất text được
    if len(doc) == 0:
        doc.close()
        return True

    total_chars = 0

    # Đếm tổng số ký tự text của toàn bộ PDF
    for page in doc:
        text = page.get_text("text") or ""
        total_chars += len(text.strip())

    # Tính số ký tự trung bình mỗi trang
    avg_chars = total_chars / len(doc)

    doc.close()

    # Nếu quá ít ký tự, đoán là PDF scan
    return avg_chars < min_chars_per_page

def is_probably_multi_column(pdf_path: str, min_x_spread: int = 180, min_blocks: int = 8) -> bool:
    doc = fitz.open(pdf_path)

    for page in doc:
        blocks = page.get_text("blocks")
        text_blocks = []

        for block in blocks:
            x0, y0, x1, y1, text, block_no, block_type = block

            if block_type != 0:
                continue

            if text and len(text.strip()) > 20:
                text_blocks.append((x0, y0, x1, y1, text))

        if len(text_blocks) >= min_blocks:
            x_positions = [b[0] for b in text_blocks]
            x_spread = max(x_positions) - min(x_positions)

            if x_spread >= min_x_spread:
                doc.close()
                return True

    doc.close()
    return False

In [4]:
def extract_text_by_simple(pdf_path: Path) -> str: #Trích xuất văn bản từ PDF bằng cách sử dụng phương pháp đơn giản
    doc = fitz.open(pdf_path)
    pages_text = [] #danh sách để lưu văn bản của từng trang
    for page in doc:
        text = page.get_text("text") #trích xuất văn bản từ trang hiện tại
        if text:
            pages_text.append(text) #nếu có văn bản, thêm vào danh sách

    doc.close()
    return normalize_pdf_text("\n".join(pages_text)) #nối tất cả văn bản của các trang lại với nhau và chuẩn hóa trước khi trả về

def extract_text_by_blocks(pdf_path: Path) -> str: #Trích xuất văn bản từ PDF bằng cách sử dụng phương pháp trích xuất khối văn bản
    doc = fitz.open(pdf_path)
    all_pages_text = [] #danh sách để lưu văn bản của tất cả các trang sau khi đã được xử lý

    for page in doc:
        blocks = page.get_text("blocks") #trích xuất các khối văn bản từ trang hiện tại, mỗi khối bao gồm thông tin về vị trí và nội dung văn bản

        text_blocks = []

        for block in blocks:
            x0, y0, x1, y1, text, block_no, block_type = block #tọa độ của blocks, nội dung văn bản, số thứ tự block, loại block

            # block_type = 0 thường là text
            if block_type != 0:
                continue

            text = normalize_pdf_text(text)
            if not text:
                continue

            text_blocks.append({
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "text": text
            })

        # Sort mặc định: từ trên xuống, trái sang phải
        text_blocks = sorted(text_blocks, key=lambda b: (round(b["y0"], 1), round(b["x0"], 1)))

        page_text = "\n".join(block["text"] for block in text_blocks)
        all_pages_text.append(page_text)

    doc.close()
    return normalize_pdf_text("\n\n".join(all_pages_text))

def extract_text_from_pdf(pdf_path: Path) -> str: #Hàm chính để trích xuất văn bản từ PDF, tự động chọn phương pháp phù hợp dựa trên đặc điểm của PDF
    pdf_path = str(pdf_path)
    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"File not found: {pdf_path}")
    result = {
        "text": "",
        "pdf_type": "",
        "method": "",
        "warnings": []
    }
    if is_probably_scanned_pdf(pdf_path):
        result["pdf_type"] = "scan_or_image_based"
        result["method"] = "none"
        result["warnings"].append(
            "PDF appears to be scanned/image-based. Text extraction returned too little text. OCR is required."
        )
        return result

    if is_probably_multi_column(pdf_path):
        text = extract_text_by_blocks(pdf_path)
        result["pdf_type"] = "multi_column_or_complex_layout"
        result["method"] = "pymupdf_blocks"
    else:
        text = extract_text_by_simple(pdf_path)
        result["pdf_type"] = "text_based"
        result["method"] = "pymupdf_text"

    result["text"] = text

    if len(text) < 300:
        result["warnings"].append(
            "Extracted text is short. Check whether the PDF is scanned, protected, or has complex layout."
        )

    return result

In [ ]:
pdf_path = r"C:\Users\Admin\Downloads\CV_PTTM.pdf"

extraction_result = extract_text_from_pdf(pdf_path)

print("PDF type:", extraction_result["pdf_type"])
print("Method:", extraction_result["method"])
print("Warnings:", extraction_result["warnings"])
print("-" * 80)
print(extraction_result["text"][:3000])
